# NaviMed Scheduling Agent Notebook

This notebook loads the synthetic scheduling dataset, ranks candidate appointment slots, and returns **agent-ready JSON** for booking and rescheduling workflows.

It is designed to support the NaviMed **Triage & Scheduling Agent** defined in your project materials:
- scheduling and modality recommendation in the MVP core flow
- event-driven handoff after confirmation
- realistic slot selection based on patient preferences, travel fit, and provider continuity

Grounding references:
- synthetic workflow and scheduling signals in the data model
- MVP core flow for scheduling and confirmation
- agent-to-agent event handoff model


In [ ]:
from pathlib import Path
import json
from datetime import datetime
import pandas as pd
import numpy as np

BASE = Path(".")

patients = pd.read_csv(BASE / "patients.csv")
clinics = pd.read_csv(BASE / "clinics.csv")
providers = pd.read_csv(BASE / "providers.csv")
slots = pd.read_csv(BASE / "schedule_slots.csv")
appointments = pd.read_csv(BASE / "appointments.csv")
reminders = pd.read_csv(BASE / "reminder_events.csv")
transport = pd.read_csv(BASE / "transport_context.csv")
preferences = pd.read_csv(BASE / "scheduling_preferences.csv")

for col in ["slot_start_ts", "slot_end_ts"]:
    slots[col] = pd.to_datetime(slots[col])

for col in ["appointment_ts", "created_at"]:
    appointments[col] = pd.to_datetime(appointments[col])

print("Loaded:")
for name, df in {
    "patients": patients,
    "clinics": clinics,
    "providers": providers,
    "slots": slots,
    "appointments": appointments,
    "reminders": reminders,
    "transport": transport,
    "preferences": preferences,
}.items():
    print(f"  {name:<12} {len(df):>4} rows")


In [ ]:
# Quick schema glance
display(patients.head(3))
display(preferences.head(3))
display(providers.head(3))
display(slots.head(3))
display(appointments.head(3))


## Build a scheduling feature view

We enrich open slots with:
- provider attributes
- clinic access attributes
- patient preferences
- provider continuity from prior appointments
- time-of-day fit
- travel-fit proxy from patient max travel preference vs clinic conditions

This is intentionally simple and explainable so you can show the ranking logic in a demo or slide.


In [ ]:
def time_band(ts: pd.Timestamp) -> str:
    hour = ts.hour
    if hour < 11:
        return "morning"
    elif hour < 15:
        return "midday"
    else:
        return "afternoon"

def clinic_travel_proxy(row, patient_transport_mode: str) -> int:
    # Lower is better; rough proxy only for synthetic agent ranking
    if row["modality"] == "telehealth":
        return 0
    parking_penalty = row["parking_difficulty_score"] * 4 if patient_transport_mode in {"car", "family_ride"} else 0
    transit_penalty = max(0, 6 - row["public_transport_access_score"]) * 4 if patient_transport_mode == "public_transit" else 0
    mobility_penalty = 8 if row.get("mobility_support_needed_flag", False) else 0
    return int(10 + parking_penalty + transit_penalty + mobility_penalty)

def get_patient_context(patient_id: str) -> dict:
    p = patients.loc[patients["patient_id"] == patient_id].iloc[0].to_dict()
    pref = preferences.loc[preferences["patient_id"] == patient_id].iloc[0].to_dict()
    prior = appointments.loc[appointments["patient_id"] == patient_id].copy()
    last_confirmed = prior.sort_values("appointment_ts").tail(1)
    prior_provider_id = last_confirmed["provider_id"].iloc[0] if len(last_confirmed) else None
    return {
        **p,
        **pref,
        "prior_provider_id": prior_provider_id,
    }

def build_candidate_slots(patient_id: str,
                          requested_specialty: str | None = None,
                          requested_modality: str | None = None,
                          visit_type: str | None = None,
                          after_ts: str | None = None) -> pd.DataFrame:
    ctx = get_patient_context(patient_id)
    df = slots.copy()
    df = df[df["slot_status"] == "open"].copy()
    df = df.merge(providers, on=["provider_id", "clinic_id", "specialty"], how="left", suffixes=("", "_provider"))
    df = df.merge(clinics, on="clinic_id", how="left")
    df["slot_day_part"] = df["slot_start_ts"].apply(time_band)
    df["patient_transport_mode"] = ctx["transport_mode"]
    df["mobility_support_needed_flag"] = ctx["mobility_support_needed_flag"]
    df["travel_proxy_minutes"] = df.apply(lambda r: clinic_travel_proxy(r, ctx["transport_mode"]), axis=1)

    if after_ts:
        df = df[df["slot_start_ts"] >= pd.to_datetime(after_ts)]

    target_specialty = requested_specialty or ctx["preferred_specialty"]
    target_modality = requested_modality or ctx["preferred_modality"]
    target_visit_type = visit_type

    if target_specialty:
        df = df[df["specialty"] == target_specialty]

    if target_modality and target_modality != "either":
        df = df[df["modality"] == target_modality]

    if target_visit_type:
        df = df[df["visit_type"] == target_visit_type]

    df["same_provider_match"] = (df["provider_id"] == ctx["prior_provider_id"]).astype(int)
    df["preferred_day_part_match"] = (df["slot_day_part"] == ctx["preferred_day_part"]).astype(int)
    df["preferred_modality_match"] = (df["modality"] == ctx["preferred_modality"]).astype(int) if ctx["preferred_modality"] != "either" else 1
    df["travel_fit"] = (df["travel_proxy_minutes"] <= ctx["max_travel_minutes"]).astype(int)
    soonest_rank = df["slot_start_ts"].rank(method="dense", ascending=True)
    df["soonest_score"] = 1 - ((soonest_rank - soonest_rank.min()) / max(1, (soonest_rank.max() - soonest_rank.min())))
    df["provider_new_patient_fit"] = df["accepting_new_patients_flag"].astype(int)

    # Explainable weighted score
    df["score"] = (
        0.18 * df["base_priority_score"] +
        0.16 * df["same_provider_match"] * (ctx["same_provider_preference_score"] / 5.0) +
        0.15 * df["preferred_day_part_match"] +
        0.13 * df["preferred_modality_match"] +
        0.16 * df["travel_fit"] * (ctx["location_convenience_preference_score"] / 5.0) +
        0.14 * df["soonest_score"] * (ctx["soonest_available_preference_score"] / 5.0) +
        0.08 * df["provider_new_patient_fit"]
    )

    return df.sort_values(["score", "slot_start_ts"], ascending=[False, True]).reset_index(drop=True)

def explain_slot(row: pd.Series, ctx: dict) -> list[str]:
    reasons = []
    if row["same_provider_match"] == 1:
        reasons.append("same provider")
    if row["preferred_day_part_match"] == 1:
        reasons.append(f"{ctx['preferred_day_part']} preference match")
    if ctx["preferred_modality"] == "either" or row["preferred_modality_match"] == 1:
        reasons.append("modality fit")
    if row["travel_fit"] == 1:
        reasons.append("travel fit")
    if row["provider_new_patient_fit"] == 1:
        reasons.append("accepting patient")
    reasons.append(f"base priority {row['base_priority_score']:.2f}")
    return reasons

def booking_response(patient_id: str,
                     requested_specialty: str | None = None,
                     requested_modality: str | None = None,
                     visit_type: str | None = None,
                     top_n: int = 5) -> dict:
    ctx = get_patient_context(patient_id)
    ranked = build_candidate_slots(
        patient_id=patient_id,
        requested_specialty=requested_specialty,
        requested_modality=requested_modality,
        visit_type=visit_type
    ).head(top_n)

    return {
        "patient_id": patient_id,
        "request_type": "book",
        "requested_specialty": requested_specialty or ctx["preferred_specialty"],
        "requested_modality": requested_modality or ctx["preferred_modality"],
        "visit_type": visit_type,
        "recommended_slots": [
            {
                "slot_id": row["slot_id"],
                "provider_id": row["provider_id"],
                "provider_name": row["provider_name"],
                "clinic_id": row["clinic_id"],
                "clinic_name": row["clinic_name"],
                "slot_start_ts": row["slot_start_ts"].isoformat(),
                "slot_end_ts": row["slot_end_ts"].isoformat(),
                "modality": row["modality"],
                "score": round(float(row["score"]), 3),
                "why_selected": explain_slot(row, ctx)
            }
            for _, row in ranked.iterrows()
        ]
    }

def reschedule_response(appointment_id: str, top_n: int = 5) -> dict:
    appt = appointments.loc[appointments["appointment_id"] == appointment_id].iloc[0].to_dict()
    ranked = build_candidate_slots(
        patient_id=appt["patient_id"],
        requested_specialty=None,
        requested_modality=appt["modality"],
        visit_type=appt["appointment_type"],
        after_ts=(pd.to_datetime(appt["appointment_ts"]) - pd.Timedelta(days=1)).isoformat()
    )

    ranked = ranked[ranked["slot_id"] != appt["slot_id"]].head(top_n)
    ctx = get_patient_context(appt["patient_id"])

    return {
        "patient_id": appt["patient_id"],
        "appointment_id": appointment_id,
        "request_type": "reschedule",
        "current_appointment": {
            "slot_id": appt["slot_id"],
            "appointment_ts": pd.to_datetime(appt["appointment_ts"]).isoformat(),
            "provider_id": appt["provider_id"],
            "clinic_id": appt["clinic_id"],
            "modality": appt["modality"],
            "appointment_type": appt["appointment_type"],
            "status": appt["status"]
        },
        "recommended_slots": [
            {
                "slot_id": row["slot_id"],
                "provider_id": row["provider_id"],
                "provider_name": row["provider_name"],
                "clinic_id": row["clinic_id"],
                "clinic_name": row["clinic_name"],
                "slot_start_ts": row["slot_start_ts"].isoformat(),
                "slot_end_ts": row["slot_end_ts"].isoformat(),
                "modality": row["modality"],
                "score": round(float(row["score"]), 3),
                "why_selected": explain_slot(row, ctx)
            }
            for _, row in ranked.iterrows()
        ]
    }


## Example 1: book a new appointment

Pick a patient and return the top candidate slots as JSON the scheduling agent could emit after ranking.


In [ ]:
example_book = booking_response(
    patient_id="PT0002",
    requested_specialty="primary_care",
    requested_modality="telehealth",
    visit_type="new_issue",
    top_n=5
)
print(json.dumps(example_book, indent=2))


## Example 2: reschedule an existing appointment

Use an existing appointment and produce replacement slots. This mirrors a reminder response like `Reschedule Request`.


In [ ]:
example_reschedule = reschedule_response("AP00002", top_n=5)
print(json.dumps(example_reschedule, indent=2))


## Optional: view ranked candidates as a table


In [ ]:
ranked_view = build_candidate_slots(
    patient_id="PT0002",
    requested_specialty="primary_care",
    requested_modality="telehealth",
    visit_type="new_issue"
)[[
    "slot_id", "provider_name", "clinic_name", "slot_start_ts", "modality",
    "travel_proxy_minutes", "same_provider_match", "preferred_day_part_match",
    "travel_fit", "base_priority_score", "score"
]].head(15)

display(ranked_view)


## Agent event payload pattern

This notebook returns ranking output. In your app layer, you would wrap it into an event like:

```json
{
  "event_type": "AppointmentOptionsRanked",
  "correlation_id": "cx-8821",
  "patient_id": "PT0002",
  "request_type": "book",
  "payload": { ... ranked slot response ... }
}
```

Then:
- patient selects a slot
- scheduling service writes the booking
- app publishes `AppointmentConfirmed`
- Structuring Agent activates downstream


In [ ]:
# Save example outputs for reuse in demos
output_dir = BASE / "sample_outputs"
output_dir.mkdir(exist_ok=True)

with open(output_dir / "booking_example.json", "w") as f:
    json.dump(example_book, f, indent=2)

with open(output_dir / "reschedule_example.json", "w") as f:
    json.dump(example_reschedule, f, indent=2)

print("Wrote sample outputs to", output_dir)


## Next steps for watsonx Studio

1. Upload these CSVs into your watsonx project.
2. Port the ranking functions into a notebook asset or service wrapper.
3. Expose a function/tool like `rank_slots(patient_id, requested_specialty, requested_modality, visit_type)`.
4. Let the Triage & Scheduling Agent call that tool after intent extraction.
5. Keep booking writes and event publishing in deterministic app code, not in the LLM itself.
